In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.mllib.evaluation.RankingMetrics

val spark = SparkSession.builder()
  .appName("Accuracy Evaluation")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
println("Spark ready: " + spark.version)

Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar.sha1
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar.sha1
Failed to download https://repo1.maven.org/maven2/net/sourceforge/f2j/arpack_combined_all/0.1/arpack_combined_all-0.1-sources.jar
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Fi

Spark ready: 3.5.9


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.mllib.evaluation.RankingMetrics
spark: SparkSession = org.apache.spark.sql.SparkSession@7417b7c1

In [2]:
import java.io.File

val baseDir = Seq("outputs/als-baseline", "../outputs/als-baseline")
  .map(new File(_)).find(_.isDirectory)
  .getOrElse(throw new RuntimeException("Could not find outputs/als-baseline"))

val runDir = baseDir.listFiles().filter(_.isDirectory).sortBy(_.getName).last.getPath
println(s"Using run folder: $runDir")

val recs = spark.read.parquet(s"$runDir/top-10-baseline")
  .select("userId", "movieId", "recommendationRank")
val test = spark.read.parquet(s"$runDir/test-ratings")
  .select("userId", "movieId", "rating")

println(s"Recommendation rows: ${recs.count()}")
println(s"Test rows: ${test.count()}")


Using run folder: ..\outputs\als-baseline\2026-10-20T21-59-51.381549100Z
Recommendation rows: 60400
Test rows: 202451


import java.io.File
baseDir: File = ..\outputs\als-baseline
runDir: String = "..\\outputs\\als-baseline\\2026-10-20T21-59-51.381549100Z"
recs: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 1 more field]
test: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 1 more field]

In [3]:
val predicted = recs
  .groupBy("userId")
  .agg(sort_array(collect_list(struct(col("recommendationRank").as("r"),
                                      col("movieId").as("m")))).as("s"))
  .withColumn("pred", expr("transform(s, x -> x.m)"))
  .select("userId", "pred")

val actual = test.filter(col("rating") >= 4.0)
  .groupBy("userId")
  .agg(collect_set("movieId").as("act"))

val joined = predicted.join(actual, Seq("userId")).select("pred", "act")

val rankRdd = joined.rdd.map { row =>
  (row.getSeq[Int](0).toArray, row.getSeq[Int](1).toArray)
}

val metrics = new RankingMetrics(rankRdd)
println(f"Users evaluated: ${joined.count()}")
println(f"Precision@10: ${metrics.precisionAt(10)}%.4f")
println(f"Recall@10:    ${metrics.recallAt(10)}%.4f")
println(f"NDCG@10:      ${metrics.ndcgAt(10)}%.4f")


Users evaluated: 6012
Precision@10: 0.0567
Recall@10:    0.0391
NDCG@10:      0.0513


predicted: org.apache.spark.sql.package.DataFrame = [userId: int, pred: array<int>]
actual: org.apache.spark.sql.package.DataFrame = [userId: int, act: array<int>]
joined: org.apache.spark.sql.package.DataFrame = [pred: array<int>, act: array<int>]
rankRdd: org.apache.spark.rdd.RDD[(Array[Int], Array[Int])] = MapPartitionsRDD[37] at map at cmd3.sc:14
metrics: RankingMetrics[Nothing] = org.apache.spark.mllib.evaluation.RankingMetrics@6a73b579